# QASPER Scientific QA Dataset for Section Retrieval

This notebook demonstrates the QASPER dataset, which contains 890 QA examples from scientific NLP papers with section-level annotations for retrieval-augmented reading.

**Key Features:**
- 890 unique questions across 289 papers from the QASPER dev split (Dasigi et al. 2021)
- Structured sections (Abstract, Introduction, Methods, Results, Discussion, Conclusion)
- Gold evidence section IDs for retrieval evaluation
- 81,550 section chunks from 1,585 NLP papers
- Perfect for benchmarking section-level retrieval and QA systems

This demo loads and explores a curated subset of the data.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

## Setup & Imports

We'll load required libraries for data processing and visualization.

In [ ]:
from pathlib import Path
import json
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Data Loading

Load the QASPER dataset from GitHub (with local fallback for offline use).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-023b95-within-document-term-weighting-for-scien/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load QASPER demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    
    # Local fallback
    if Path('mini_demo_data.json').exists():
        with open('mini_demo_data.json') as f:
            return json.load(f)
    
    raise FileNotFoundError('Could not load mini_demo_data.json')

# Load the data
data = load_data()
print(f'✓ Loaded data: {data["metadata"]["description"]}')
print(f'  Source: {data["metadata"]["source"]}')
print(f'  Examples: {data["metadata"]["num_examples"]}')

## Section Type Classification

Define patterns to classify paper sections (Abstract, Methods, Results, etc.) based on section names.

In [ ]:
SECTION_TYPE_MAP = {
    r'abstract': 'Abstract',
    r'introduction': 'Introduction',
    r'related.?work|background|literature': 'RelatedWork',
    r'method|approach|model|system|architecture|experiment|setup|implementation': 'Methods',
    r'result|finding|evaluation|performance|comparison|experiment': 'Results',
    r'discussion|analysis|ablation|error': 'Discussion',
    r'conclusion|future|summary|limitation': 'Conclusion',
}

def infer_section_type(name: str) -> str:
    """Infer section type from section name using regex patterns."""
    if not name:
        return 'Other'
    n = name.lower()
    for pattern, stype in SECTION_TYPE_MAP.items():
        if re.search(pattern, n):
            return stype
    return 'Other'

print('✓ Section type inference loaded')

## Extract and Process Examples

Extract QA examples from the loaded data structure, parse metadata, and build dataframes for analysis.

In [ ]:
# Extract examples from the data structure
examples = data['datasets'][0]['examples']
print(f'✓ Extracted {len(examples)} examples')

# Parse metadata and build records for analysis
records = []
for ex in examples:
    # Parse JSON fields
    sections_list = json.loads(ex.get('metadata_sections_json', '[]'))
    evidence_ids = json.loads(ex.get('metadata_evidence_section_ids', '[]'))
    evidence_types = json.loads(ex.get('metadata_evidence_section_types', '[]'))
    
    record = {
        'query_id': ex.get('metadata_query_id', ''),
        'doc_title': ex.get('metadata_doc_title', ''),
        'paper_id': ex.get('metadata_paper_id', ''),
        'num_sections': ex.get('metadata_num_sections', 0),
        'split_source': ex.get('metadata_split_source', 'other'),
        'num_evidence_sections': len(evidence_ids),
        'evidence_types': ', '.join(evidence_types),
        'question_length': len(ex.get('input', '')),
        'answer_length': len(ex.get('output', '')),
        'question': ex.get('input', '')[:100] + '...',
        'answer': ex.get('output', '')[:100] + '...',
    }
    records.append(record)

# Create DataFrame
df = pd.DataFrame(records)
print(f'\n✓ Created DataFrame with {len(df)} rows')
print(f'\nColumns: {list(df.columns)}')

## Dataset Statistics

Compute key statistics about the dataset: paper distribution, section coverage, and evidence characteristics.

In [ ]:
# Compute statistics
stats = {
    'Total Examples': len(df),
    'Unique Papers': df['paper_id'].nunique(),
    'Avg Sections/Paper': df['num_sections'].mean(),
    'Avg Evidence Sections': df['num_evidence_sections'].mean(),
    'Avg Question Length': df['question_length'].mean(),
    'Avg Answer Length': df['answer_length'].mean(),
}

print('\n' + '='*50)
print('DATASET STATISTICS')
print('='*50)
for key, value in stats.items():
    if isinstance(value, float):
        print(f'{key:.<30} {value:.2f}')
    else:
        print(f'{key:.<30} {value}')

print('\n' + '='*50)
print('SPLIT SOURCE DISTRIBUTION')
print('='*50)
split_dist = df['split_source'].value_counts()
for split, count in split_dist.items():
    pct = 100.0 * count / len(df)
    print(f'{split:.<30} {count:>3} ({pct:>5.1f}%)')

print('\n' + '='*50)
print('EVIDENCE TYPES')
print('='*50)
all_evidence_types = []
for types_str in df['evidence_types']:
    if types_str:
        all_evidence_types.extend(t.strip() for t in types_str.split(','))
evidence_counts = pd.Series(all_evidence_types).value_counts()
for etype, count in evidence_counts.items():
    pct = 100.0 * count / len(all_evidence_types)
    print(f'{etype:.<30} {count:>3} ({pct:>5.1f}%)')

## Sample Examples

Inspect individual QA examples to see question, answer, and section context.

In [ ]:
# Display first example in detail
print('\n' + '='*70)
print('SAMPLE EXAMPLE #0')
print('='*70)

ex = examples[0]
print(f'\nPaper: {ex["metadata_doc_title"]}')
print(f'Paper ID: {ex["metadata_paper_id"]}')
print(f'Total Sections: {ex["metadata_num_sections"]}')

print(f'\n--- QUESTION ---')
qtext = ex['input']
if len(qtext) > 500:
    print(qtext[:500] + '...')
else:
    print(qtext)

print(f'\n--- ANSWER ---')
print(ex['output'])

evidence_ids = json.loads(ex['metadata_evidence_section_ids'])
evidence_types = json.loads(ex['metadata_evidence_section_types'])
print(f'\n--- EVIDENCE SECTIONS ---')
print(f'Section IDs: {evidence_ids}')
print(f'Section Types: {evidence_types}')

print(f'\n--- ABSTRACT (first 300 chars) ---')
abs_text = ex['metadata_doc_abstract']
if len(abs_text) > 300:
    print(abs_text[:300] + '...')
else:
    print(abs_text)

print('\n' + '='*70)

## Visualization

Create charts to visualize dataset composition and key metrics.

In [ ]:
# Set up plotting style
plt.style.use('default')
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('QASPER Dataset Overview', fontsize=14, fontweight='bold')

# Plot 1: Section Count Distribution
ax = axes[0, 0]
df['num_sections'].hist(bins=15, ax=ax, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('Number of Sections')
ax.set_ylabel('Frequency')
ax.set_title('Sections per Paper')
ax.grid(axis='y', alpha=0.3)

# Plot 2: Split Source Distribution
ax = axes[0, 1]
split_counts = df['split_source'].value_counts()
split_counts.plot(kind='barh', ax=ax, color='coral', edgecolor='black', alpha=0.7)
ax.set_xlabel('Number of Examples')
ax.set_title('Evidence Type Distribution')
ax.grid(axis='x', alpha=0.3)

# Plot 3: Question vs Answer Length
ax = axes[1, 0]
ax.scatter(df['question_length'], df['answer_length'], alpha=0.6, color='green', s=100, edgecolor='black')
ax.set_xlabel('Question Length (chars)')
ax.set_ylabel('Answer Length (chars)')
ax.set_title('Question vs Answer Length')
ax.grid(alpha=0.3)

# Plot 4: Evidence Sections Distribution
ax = axes[1, 1]
df['num_evidence_sections'].hist(bins=10, ax=ax, color='purple', edgecolor='black', alpha=0.7)
ax.set_xlabel('Number of Evidence Sections')
ax.set_ylabel('Frequency')
ax.set_title('Evidence Sections per Query')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('✓ Visualization complete')

## Summary

This demo notebook successfully loaded and analyzed the QASPER scientific QA dataset. The data consists of:

- **890 QA examples** from the QASPER dev split (Dasigi et al. 2021)
- **276 unique papers** with full section annotations
- **Section-level evidence** for retrieval evaluation
- **Rich metadata** including paper titles, abstracts, section lists, and answer text

The dataset is designed for benchmarking section-level retrieval systems that rank paper sections by query relevance and extract answers from retrieved sections.

**Next steps:**
- Implement a retrieval ranker (BM25, dense embedding model, etc.)
- Evaluate top-k section retrieval accuracy
- Measure answer F1 scores from retrieved section context
- Benchmark against baseline methods